In [72]:
import pandas as pd
import re 
from rdkit import Chem

In [73]:
from rdkit import Chem
import re

def formula_to_dummy_smiles(formula):

    if 'C' in formula:
        carbon_count = int(re.search(r'C(\d+)', formula).group(1))
        return 'C'*carbon_count
 
formula_to_dummy_smiles('C18H32O2SR')

'CCCCCCCCCCCCCCCCCC'

In [74]:
from cobra.io import read_sbml_model
import cobra

model = read_sbml_model("photo_model_last.xml")

columns = ['#reaction_ID', 'reactant_IDs(atom)', 'product_IDs(atom)', 'reversibility']
df = pd.DataFrame(columns=columns)

met_list = [met.id for met in kokel.metabolites]
rxn_list = [rxn.id for rxn in kokel.reactions]

#print(met_list)
#print(rxn_list)

Sm = cobra.util.array.create_stoichiometric_matrix(kokel)

No objective in listOfObjectives
No objective coefficients in model. Unclear what should be optimized


In [75]:
print(met_list)

['rb15bp[h]', 'co2[h]', '3pg[h]', 'dhap[h]', 'e4p[h]', 's17bp[h]', 's7p[h]', 'fdp_B[h]', 'f6p_B[h]', 'xu5p_D[h]', 'r5p[h]', 'ru5p_D[h]', '2pglyc[h]', 'glyclt[h]', 'glx[h]', 'glx[c]', 'ser_L[h]', 'ser_L[c]', 'g6p_A[h]', 'g1p[h]', 'adpglc[h]', 'dhap[c]', 'fdp_B[c]', 'f6p_B[c]', 'g6p_A[c]', 'g1p[c]', 'udpg[c]', '3pg[c]', 'pep[c]', 'pyr[c]', 'co2[c]', 'oaa[c]', 'ala_L[c]', 'mal_L[c]', 'asp_L[c]', 'akg[c]', 'glu_L[c]', 'pep[h]', 'co2[e]', 'pyr[h]', 'pyr[m]', 'aca[m]', 'co2[m]', 'oaa[m]', 'cit[m]', 'icit[m]', 'akg[m]', 'sca[m]', 'succ[m]', 'fum[m]', 'mal_L[m]', 'asp_L[m]', 'glu_L[m]', 'ala_L[m]', 'glx[m]', 'pep[m]']


In [76]:
metabolite_smiles = dict()
metabolite_inchi = dict()

In [77]:
ref = pd.read_csv('elements.csv')

def add_compartment(compound):
    if not re.search(r'\[[a-z]\]$', compound):  # Check if compartment is missing
        return f"{compound}[c]"
    return compound

#### Apply to the column
ref['Abbreviation'] = ref['Abbreviation'].apply(add_compartment)
ref['Abbreviation'] = ref['Abbreviation'].str.replace('-', '_')

filtered_df = ref[ref['Abbreviation'].isin(met_list)]
filtered_df.to_excel('essentmets_new.xlsx')
filtered_df

,Abbreviation,Compartment,Name,Formula,Charge,CAS Number,Formula Neutral,KEGG cmpd ID,PubChem Substance ID
101,2pglyc[h],Chloroplast,2-Phosphoglycolate,C2H2O6P,-3,,C2H5O6P,C00988,4234
185,3pg[c],Cytosol,3-Phospho-D-glycerate,C3H4O7P,-3,,C3H7O7P,C00197,3497
187,3pg[h],Chloroplast,3-Phospho-D-glycerate,C3H4O7P,-3,,C3H7O7P,C00197,3497
340,adpglc[h],Chloroplast,ADPglucose,C16H23N5O15P2,-2,,C16H25N5O15P2,C00498,3781
356,akg[c],Cytosol,2-Oxoglutarate,C5H4O5,-2,328-50-7,C5H6O5,C00026,3328
358,akg[m],Mitochondria,2-Oxoglutarate,C5H4O5,-2,328-50-7,C5H6O5,C00026,3328
364,ala_L[c],Cytosol,L-Alanine,C3H7NO2,0,56-41-7,C3H7NO2,C00041,3343
365,ala_L[m],Mitochondria,L-Alanine,C3H7NO2,0,56-41-7,C3H7NO2,C00041,3343
440,asp_L[c],Cytosol,L-Aspartate,C4H6NO4,-1,56-84-8,C4H7NO4,C00049,3351
442,asp_L[m],Mitochondria,L-Aspartate,C4H6NO4,-1,56-84-8,C4H7NO4,C00049,3351


In [78]:
import requests
import pubchempy as pcp
from time import sleep
import cv2
import numpy as np
from io import BytesIO
from PIL import Image
import re
from xml.etree import ElementTree as ET
from bs4 import BeautifulSoup  # Add this for web scraping

def get_smiles_from_id(kegg_id=None, pubchem_sid=None, formula=None, max_retries=3):
    # --- PubChem SID Lookup Function ---
    def fetch_pubchem_smiles(sid):
        for attempt in range(max_retries):
            try:
                sid = int(float(pubchem_sid))
                substance = pcp.Substance.from_sid(sid)
                if substance.standardized_compound:
                    return substance.standardized_compound.canonical_smiles
                return None
            except Exception as e:
                if attempt < max_retries - 1:
                    sleep(2 ** attempt)
                else:
                    print(f"PubChem SID lookup failed for {sid}: {str(e)}")
        return None

    # --- ChEBI Lookup Function (via KEGG cross-reference) ---
    def fetch_chebi_smiles_from_kegg(kegg_id):
        # Step 1: Get ChEBI ID from KEGG
        kegg_url = f"http://rest.kegg.jp/get/compound:{kegg_id}"
        for attempt in range(max_retries):
            try:
                response = requests.get(kegg_url, timeout=10)
                if response.status_code == 200:
                    kegg_data = response.text
                    chebi_match = re.search(r'ChEBI:\s*(\d+)', kegg_data)
                    if chebi_match:
                        chebi_id = f"CHEBI:{chebi_match.group(1)}"
                    else:
                        print(f"No ChEBI cross-reference found for KEGG ID {kegg_id}")
                        return None
                else:
                    print(f"KEGG API request failed for {kegg_id}: Status {response.status_code}")
                    return None
                break
            except Exception as e:
                if attempt < max_retries - 1:
                    sleep(2 ** attempt)
                else:
                    print(f"Failed to fetch KEGG data for {kegg_id}: {str(e)}")
                    return None
    # --- Execution Logic ---
    if pubchem_sid and pubchem_sid != '':
        try:
            sid = int(pubchem_sid) if str(pubchem_sid).isdigit() else pubchem_sid
            print(sid)
            smiles = fetch_pubchem_smiles(sid)
            if smiles:
                return smiles
        except ValueError:
            print(f"Invalid PubChem SID format: {pubchem_sid}")

    if kegg_id and kegg_id != '':
        smiles = fetch_chebi_smiles_from_kegg(kegg_id)
        if smiles:
            return smiles

    
    return sus

In [84]:
buffer = filtered_df.copy()
for index, row in buffer.iterrows():

    if row['Abbreviation'] not in metabolite_smiles.keys():
        result = get_smiles_from_id(row['KEGG cmpd ID'], row['PubChem Substance ID'], row['Formula']) 
        metabolite_smiles[row['Abbreviation']] = result

metabolite_smiles['aca[m]'] = "CC"
metabolite_smiles['sca[m]'] = "CC"

In [85]:
from rdkit import Chem
from rdkit.Chem import inchi
from rdkit import RDLogger

for compound, smiles in metabolite_smiles.items():

    try:
        molecule = Chem.MolFromSmiles(smiles)
        # Generate InChI
        inchi_string = inchi.MolToInchi(molecule)
        print(compound)
        # Generate InChI Key
        inchi_key = inchi.InchiToInchiKey(inchi_string)
        print(inchi_key)
        print('##################################################')

        RDLogger.DisableLog('rdApp.warning')
        #print(f"{compound}\t{inchi_key}")
        metabolite_inchi[compound] = inchi_key
        metabolite_smiles[compound] = smi_fixed
    except Exception as e: 
        continue

2pglyc[h]
ASCFNMCAHFUBCO-UHFFFAOYSA-N
##################################################
3pg[c]
OSJPPGNTCRNQQC-UHFFFAOYSA-N
##################################################
3pg[h]
OSJPPGNTCRNQQC-UHFFFAOYSA-N
##################################################
adpglc[h]
WFPZSXYXPSUOPY-UHFFFAOYSA-N
##################################################
akg[c]
KPGXRSRHYNQIFN-UHFFFAOYSA-N
##################################################
akg[m]
KPGXRSRHYNQIFN-UHFFFAOYSA-N
##################################################
ala_L[c]
QNAYBMKLOCPYGJ-UHFFFAOYSA-N
##################################################
ala_L[m]
QNAYBMKLOCPYGJ-UHFFFAOYSA-N
##################################################
asp_L[c]
CKLJMWTZIZZHCS-UHFFFAOYSA-N
##################################################
asp_L[m]
CKLJMWTZIZZHCS-UHFFFAOYSA-N
##################################################
cit[m]
KRKNYBCHXYNGOX-UHFFFAOYSA-N
##################################################
co2[c]
CURLTUGMZLYLDI-UHFFFAOYSA-N
#####

In [87]:
import os
import numpy as np

# Construct a stoichiometric matrix

S_df = pd.DataFrame(Sm, met_list, rxn_list)

failed_rxns = list()

parent_folder = '/home/duford/Finito/mapping/reaction_intermediates3'
folder_path = '/home/duford/Finito/mapping/reaction_intermediates3/'


for rxn in S_df.columns:
    
    reactants = list()
    products = list()

    col_data = S_df[rxn]
    
    non_zero_mask = col_data != 0
    non_zero_mets = col_data[non_zero_mask].index
    

    if len(non_zero_mets) > 0:
        for met in non_zero_mets:
            coeff = col_data[met]
            
            if coeff < 0:
                for i in range(abs(int(coeff))):
                    reactants.append(met) 
            else:
                for i in range(abs(int(coeff))):
                    products.append(met)
                               
            role = "consumed" if coeff < 0 else "produced"

    reactantSM = [metabolite_smiles[r] for r in reactants]
    productSM = [metabolite_smiles[p] for p in products] 
    
    SM = reactantSM + productSM

    compounds = reactants + products
    
    if any('C' in compound for compound in SM) == False:
        continue
    if len(reactantSM) == 0 or len(productSM) == 0:
        continue

    os.makedirs(os.path.join(parent_folder, rxn), exist_ok=True)
    
    reactants_merged = ".".join(reactantSM)
    products_merged = ".".join(productSM)

    carbon_react = reactants_merged.upper().count("C")
    carbon_prod = products_merged.upper().count("C")

    fullR = reactants_merged + ">>" + products_merged

    file_name_smiles = "rxn.smiles"       # File name
    file_path_smiles = os.path.join(f'{folder_path}{rxn}', file_name_smiles)

    file_name_reactants = "from_species_with_cmp"
    file_path_reactants = os.path.join(f'{folder_path}{rxn}', file_name_reactants)

    file_name_products = "to_species_with_cmp"
    file_path_products = os.path.join(f'{folder_path}{rxn}', file_name_products)

    file_name_inchi = "species_id_inchikey.txt"
    file_path_inchi = os.path.join(f'{folder_path}{rxn}', file_name_inchi)
    
    
    with open(file_path_smiles, "w") as file:
        file.write(fullR)

    with open(file_path_reactants, "w") as file:
        for reactant in reactants:
            file.write(reactant + "\n")

    with open(file_path_products, "w") as file:
        for product in products:
            file.write(product + "\n")
    
    with open(file_path_inchi, "w") as file:
        for compound in compounds:
            file.write(f"{compound}\t{metabolite_inchi[compound]}\n")